In [ ]:
%run ../module7-genai-langchain/initialize_environment.py

## Local MCP server

In [ ]:
client =  MultiServerMCPClient(
        {
            "local_server": {
                    "transport": "stdio",
                    "command": "python",
                    "args": ["resources/mcp_server_basic.py"],
                }
        }
    )

server = "local_server"

# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources(server)

# get prompts
prompt = await client.get_prompt(server, "prompt")

UnsupportedOperation: fileno

In [ ]:
llm = create_azure_llm()

agent = create_agent(
    llm,
    tools=tools,
    system_prompt=prompt[0].content
)

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

response

## Connect to an online external/3rd Party MCP Servers

MCP Time Server: [https://mcpservers.org/servers/modelcontextprotocol/time](https://mcp.so/server/time/modelcontextprotocol)

### What is `uvx`?

`uvx` is a command-line tool from the [uv](https://github.com/astral-sh/uv) project that allows you to run Python packages directly without installing them globally or in a virtual environment. It's similar to `npx` in Node.js. When you run `uvx package-name`, it:

1. Downloads the package if not already cached
2. Creates an isolated environment
3. Executes the package's command-line interface
4. Cleans up after execution

This is particularly useful for running standalone tools and MCP servers without affecting your existing Python environment.
In this notebook, the MCP client starts the server with:

- command: `uvx`
- args: `mcp-server-time --local-timezone=America/New_York`

This means `uvx` resolves and runs the `mcp-server-time` package executable, and the client communicates with it over standard input/output as an MCP transport.

In [ ]:
time_client = MultiServerMCPClient(
    {
        # Below JSON is available directly from the mcp server link above:
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

time_tools = await time_client.get_tools()

agent = create_agent(
    llm,
    tools=time_tools,
)

question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)